# Event order books

Track full bid/ask depth and available shares for every outcome in selected markets. Dates are market metadata; update timestamps describe when quotes changed. Select the repository `.venv` kernel. Network calls are disabled until you set the flags below.


In [ ]:
from pathlib import Path
import sys

repo = Path.cwd()
if not (repo / "src" / "polytrader").exists():
    repo = repo.parent
if not (repo / "src" / "polytrader").exists():
    raise RuntimeError("Run this notebook from the repository root or sandbox directory")
sys.path.insert(0, str(repo / "src"))

from polytrader.orderbook import fetch_orderbooks, watch_orderbooks


In [ ]:
EVENT = "<event-slug>"
MARKETS = None  # Or ["<nov-market-slug>", "<dec-market-slug>"] in your preferred order.
OUTCOMES = None  # Or ["Yes"]. Default: all outcomes, including nonbinary labels.
RUN_SNAPSHOT = False
RUN_LIVE = False


## Snapshot and quantities

Set `EVENT`, then enable `RUN_SNAPSHOT`. Each book has sorted price levels with `price` and `size` (shares). Missing bid/ask sides have no spread. Inspect status and exclusion reasons before using the data.


In [ ]:
books = None
if RUN_SNAPSHOT:
    books = fetch_orderbooks(EVENT, markets=MARKETS, outcomes=OUTCOMES)
    print("Excluded markets:", books.excluded)
    import pandas as pd
    rows = books.summary()
    frame = pd.DataFrame(rows)
    if not frame.empty:
        display(frame[["market_slug", "outcome", "status", "best_bid", "best_ask", "spread", "updated_at"]])
else:
    print("Set EVENT and RUN_SNAPSHOT=True to fetch current books.")


In [ ]:
if books and books.books:
    token = next(iter(books.books))
    book = books.books[token]
    print(books.markets[token].question, books.markets[token].outcome, book.status)
    display(pd.DataFrame([level.to_dict() for level in book.bids[:10]]))
    display(pd.DataFrame([level.to_dict() for level in book.asks[:10]]))


## Bounded live stream

Install `python -m pip install -e ".[live]"` from the repository root. Enable `RUN_LIVE` to watch for up to 15 seconds. The context manager closes the background connection even on timeout or interruption. A slow consumer receives the latest notification per token; this is not a historical recorder. Restart the stream after changing the selected event or dates.


In [ ]:
import asyncio
from contextlib import aclosing

async def sample_live(seconds=15):
    latest = {}
    try:
        async with asyncio.timeout(seconds):
            async with aclosing(watch_orderbooks(EVENT, markets=MARKETS, outcomes=OUTCOMES)) as updates:
                async for update in updates:
                    latest[update.book.token_id] = update.to_dict()
                    print(update.market_slug, update.outcome, update.book.status, update.book.spread)
    except TimeoutError:
        print("Sample finished; connection closed.")
    return latest

if RUN_LIVE:
    latest = await sample_live()
else:
    print("Set RUN_LIVE=True to stream public updates.")


## Compact recording and offline replay
Set `RUN_RECORDING=True` to collect a 30-second sample. Only changes to best bid/ask, sizes, and status are saved; full depth is omitted. The file has a 1 MiB cap and is never overwritten. Set `RECORDING_PATH` to an existing recording to replay without any network requests.


In [ ]:
from datetime import datetime, timezone
from polytrader.orderbook import record_orderbooks, load_recording, replay_orderbooks

RUN_RECORDING = False
RECORDING_PATH = None  # Or Path("data/orderbooks/your-recording.jsonl").
if RUN_RECORDING:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
    RECORDING_PATH = repo / "data" / "orderbooks" / f"quotes-{stamp}.jsonl"
    result = await record_orderbooks(EVENT, markets=MARKETS, outcomes=OUTCOMES,
                                    output=RECORDING_PATH, duration=30, interval=1,
                                    max_bytes=1024 * 1024)
    print(result)


In [ ]:
if RECORDING_PATH is not None:
    recording = load_recording(RECORDING_PATH)
    print(recording.metadata["event"])
    # Immediate offline inspection; .frames() streams without loading the whole file.
    for frame in recording.frames():
        for token, quote in frame.quotes.items():
            print(frame.recorded_at, token[-12:], quote.status, quote.spread)
    # Timed replay: async for frame in replay_orderbooks(recording, speed=10): ...
